# Multi-Node Heat Network

A central plant supplies heat to a remote district through a pipeline with
limited capacity and heat loss. When the pipeline saturates, a local backup
boiler covers the remaining demand.

**New concepts**: Multi-node carriers (`node`) · Transmission between nodes · Pipeline losses

In [ ]:
import math
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.io as pio

from fluxopt import Carrier, Converter, Effect, Flow, Port, optimize

pio.renderers.default = 'notebook_connected'

## System

```
Node A (plant)                          Node B (district)

Gas Grid ─► Boiler η=92%                   Backup η=85%
 0.05 €/kWh     │                        0.12 €/kWh  │
            [heat:A] ─── Pipeline ───► [heat:B] ──► District
                       60 kW, η=95%
```

- **Node A** has a cheap gas boiler at the central plant
- **Node B** has the residential demand and an expensive oil backup
- The **pipeline** transmits up to 60 kW from A to B with 5% heat loss
- When demand exceeds pipeline capacity, the backup kicks in

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]

# Residential demand: morning + evening peaks, 30 kW baseload
demand = [
    30 + 50 * max(0.0, math.sin(math.pi * ((h % 24) - 5) / 6)) + 40 * max(0.0, math.sin(math.pi * ((h % 24) - 16) / 6))
    for h in range(n)
]
max_d = max(demand)

In [ ]:
max_d = max(demand)

# Pipeline flows — referenced in conversion_factors
pipe_in = Flow('heat', node='A', size=60)
pipe_out = Flow('heat', node='B', size=60)

result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('oil'), Carrier('heat', nodes=['A', 'B'])],
    effects=[Effect('cost', unit='EUR')],
    ports=[
        Port('gas_grid', imports=[Flow('gas', size=200, effects_per_flow_hour={'cost': 0.05})]),
        Port('oil_supply', imports=[Flow('oil', size=200, effects_per_flow_hour={'cost': 0.12})]),
        Port(
            'district',
            exports=[
                Flow('heat', node='B', size=max_d, fixed_relative_profile=[d / max_d for d in demand]),
            ],
        ),
    ],
    converters=[
        Converter.boiler('central_boiler', 0.92, Flow('gas', size=200), Flow('heat', node='A', size=100)),
        # Pipeline: heat from node A to node B with 5% loss
        Converter(
            'pipeline', inputs=[pipe_in], outputs=[pipe_out], conversion_factors=[{'heat:A': 0.95, 'heat:B': -1}]
        ),
        Converter.boiler('backup_boiler', 0.85, Flow('oil', size=200), Flow('heat', node='B', size=100)),
    ],
    objective_effects='cost',
)

print(f'Total cost: {result.objective:.2f} EUR  |  Avg: {result.objective / sum(demand) * 100:.2f} ct/kWh')

## Results

In [ ]:
times = result.flow_rates.coords['time'].values
pipeline = result.flow_rate('pipeline(heat:B)').values
backup = result.flow_rate('backup_boiler(heat:B)').values

fig = go.Figure()
fig.add_trace(go.Scatter(x=times, y=pipeline, name='pipeline (from A)', line_shape='hv', stackgroup='s'))
fig.add_trace(go.Scatter(x=times, y=backup, name='backup (local)', line_shape='hv', stackgroup='s'))
fig.add_trace(go.Scatter(x=times, y=demand, name='demand', mode='markers', marker={'color': 'black', 'size': 4}))
fig.update_layout(
    title='Heat Supply at Node B',
    yaxis_title='kW',
    height=350,
    margin={'l': 50, 'r': 20, 't': 40, 'b': 20},
    template='plotly_white',
)
fig

## Insights

- **Pipeline-first dispatch**: the optimizer maximizes cheap central supply through the pipeline
- **Backup only at peaks**: the expensive local boiler only runs when demand exceeds pipeline capacity (~57 kW delivered, 60 kW input at 95%)
- **Transmission loss**: 5% of heat is lost in the pipeline — the central boiler produces more than what arrives at B
- **Use case**: district heating networks, industrial sites with remote process heat, campus energy systems